# Treinamento de Modelo BERT para Detecção de Anatomo - Enhanced MLflow Tracking

Este notebook implementa o treinamento de um modelo BERTimbal para classificação binária de laudos médicos,
identificando a presença de anatomo, com sistema completo de rastreamento MLflow.

## 1. Instalação de Dependências

In [0]:
!pip install unidecode
!pip install torch
!pip install transformers
!pip install openpyxl
!pip install cloudpickle==2.2.1

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

## 2. Importação de Bibliotecas

In [0]:
import os
import platform
import random
import sys
import time
import re
from datetime import datetime
from copy import deepcopy
import cloudpickle
import matplotlib.pyplot as plt
import mlflow
import mlflow.data
import mlflow.pyfunc
import mlflow.pytorch
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler

import transformers
from transformers import BertTokenizer, BertForSequenceClassification, AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from mlflow.models.signature import infer_signature
from mlflow.pyfunc import PythonModel, PythonModelContext
from mlflow.tracking import MlflowClient

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    auc
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from unidecode import unidecode
from tqdm import tqdm

2026-02-06 12:25:02.657469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-06 12:25:02.671878: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-06 12:25:02.676246: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-06 12:25:02.688452: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-06 12:25:06.090144: W tensorflow/compiler/tf2

## 3. Configuração de Parâmetros do Experimento

In [0]:
model_name = "ANATOMOPATHOLOGICALS"

# Versão da tabela do dataset
dataset_version = 2
dataset_table_name = "datascience_prd.gold.dataset_anatomo_v2"

# Seed para reprodutibilidade
random_seed = 42

# Divisão de dados
test_size = 0.2
use_validation = False
validation_size = 0.5

# Timestamp para identificação
formatted_datetime = datetime.now().strftime("%d_%m_%y_%H%M%S")
experiment_name = model_name + f"_{formatted_datetime}"

## 4. Coleta de Informações do Ambiente

In [0]:
# Informações do sistema operacional
os_name = os.name
system_name = platform.system()
system_version = platform.version()
release = platform.release()
architecture = platform.architecture()
python_version = sys.version

print(f"os_name: {os_name}")
print(f"system_name: {system_name}")
print(f"system_version: {system_version}")
print(f"release: {release}")
print(f"architecture: {architecture}")
print(f"python_version: {python_version}")

os_name: posix
system_name: Linux
system_version: #100-Ubuntu SMP Tue May 27 21:41:06 UTC 2025
release: 5.15.0-1091-azure
architecture: ('64bit', 'ELF')
python_version: 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]


In [0]:
# Informações das bibliotecas
numpy_version = np.__version__
pandas_version = pd.__version__
scikit_learn_version = sklearn.__version__
mlflow_version = mlflow.__version__
pytorch_version = torch.__version__
pytorch_version_cuda = torch.version.cuda
pytorch_version_cudnn = torch.backends.cudnn.version() if torch.cuda.is_available() else None
transformers_version = transformers.__version__
cloudpickle_version = cloudpickle.__version__
cuda_available = torch.cuda.is_available()
cuda_version = torch.version.cuda
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"numpy_version: {numpy_version}")
print(f"pandas_version: {pandas_version}")
print(f"scikit_learn_version: {scikit_learn_version}")
print(f"mlflow_version: {mlflow_version}")
print(f"pytorch_version: {pytorch_version}")
print(f"pytorch_version_cuda: {pytorch_version_cuda}")
print(f"pytorch_version_cudnn: {pytorch_version_cudnn}")
print(f"transformers_version: {transformers_version}")
print(f"cuda_available: {cuda_available}")
print(f"cuda_version: {cuda_version}")
print(f"device: {device}")

numpy_version: 1.26.4
pandas_version: 2.3.3
scikit_learn_version: 1.4.2
mlflow_version: 3.8.1
pytorch_version: 2.6.0+cu124
pytorch_version_cuda: 12.4
pytorch_version_cudnn: 90100
transformers_version: 4.50.2
cuda_available: True
cuda_version: 12.4
device: cuda


## 5. Configuração de Seed para Reprodutibilidade

In [0]:
def set_seed(seed=42):
    """Fixa a seed para garantir reprodutibilidade"""
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(random_seed)
print(f"random_seed: {random_seed}")

random_seed: 42


## 6. Hiperparâmetros do Modelo

In [0]:
# Lista de modelos foundation para testar
foundation_model_name = "neuralmind/bert-base-portuguese-cased"  # 512 tokens

# Hiperparâmetros
epochs = 2
batch_size = 32
learning_rate = 1e-5
token_max_length = 512

# Configurações adicionais
use_dropout = True
hidden_dropout = 0.1
attention_dropout = 0.1
use_scheduler = False
use_weights = False

# Early stopping
patience = 3
min_delta = 0.001

print("Hiperparâmetros configurados:")
print(f"  Epochs: {epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {learning_rate}")
print(f"  Max token length: {token_max_length}")

Hiperparâmetros configurados:
  Epochs: 2
  Batch size: 32
  Learning rate: 1e-05
  Max token length: 512


## 7. Funções de Pré-processamento

In [0]:
def preprocess_bert(text):
    if isinstance(text, str):
        processed_text = text.lower()
        processed_text = unidecode(processed_text)
        # FIXME: use an actual html encoder
        processed_text = processed_text.replace("<br>", " ")
        processed_text = processed_text.replace("\t", " ")
        processed_text = processed_text.replace(";", ".")
        processed_text = processed_text.replace("nasc.", "nasc")
        processed_text = processed_text.replace("dr.", "dr")
        processed_text = processed_text.replace("dta.", "data")
        processed_text = processed_text.replace(" /*", ". ")
        processed_text = processed_text.replace("*/ ", ". ")
        processed_text = processed_text.replace("_", "")
        processed_text = processed_text.replace(". . ", ". ")
        processed_text = processed_text.replace("..", ". ")
        processed_text = processed_text.replace("\n", " ")
        processed_text = processed_text.replace(".", " ")
        processed_text = processed_text.replace("/", " ")
        processed_text = processed_text.replace("(", " ")
        processed_text = processed_text.replace(")", " ")
        processed_text = processed_text.replace("{", " ")
        processed_text = processed_text.replace("}", " ")
        processed_text = processed_text.replace(">", " ")
        processed_text = processed_text.replace("<", " ")
        processed_text = processed_text.replace("-", " ")
        processed_text = processed_text.replace(" : ", ": ")
        processed_text = processed_text.replace("=", " ")
        processed_text = processed_text.replace(":", " ")
        processed_text = re.sub("\s\s+", " ", processed_text)
        processed_text = re.sub(r"[0-9]+", " ", processed_text)
        processed_text = re.sub(" +", " ", processed_text)
        processed_text = processed_text.split("anteriores realizados")[0]
    else:
        processed_text = ""

    return processed_text



def df_downsample(df, col:str, rand_state: bool = True, rand_value:int=42):
    """
    Description:
        Função que realiza balanceamento para a mínima frequência (downsample) a partir da coluna de um dataframe. 
    Args:
        df  - (pandas.DataFrame) pandas DataFrame com objetivo de realizar o downsample.
        col - (str) coluna objetivo do downsample.
        rand_state - (True/False) variável que identifica se a amostra é aleatorizada totalmente ou parcialmente (indicando um random_state fixo no sample). Recebe binário (True ou False)
    Return:
        df_strat - pandas DataFrame estratificado resultante do downsample.
    """
    col = str(col)
    val_counts = df[col].value_counts()
    freq_min = val_counts.min()
    
    if rand_state:
        df_strat = df.groupby(col, group_keys=False).apply(
            lambda x: x.sample(freq_min, random_state=rand_value)
        )
    else:
        df_strat = df.groupby(col, group_keys=False).apply(lambda x: x.sample(freq_min))
        
    return df_strat

def clean_string(text):
    """Substitui caracteres especiais por underline"""
    return re.sub(r'[^A-Za-z0-9_]', '_', text)

<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
/root/.ipykernel/4840/command-6769216178717841-1107626603:30: SyntaxWarning: invalid escape sequence '\s'
  processed_text = re.sub("\s\s+", " ", processed_text)


## 8. Classe de Classificação BERT com MLflow

In [0]:
class BERTBinaryClassification:
    """
    Modelo BERT para classificação binária com integração completa ao MLflow.
    """
    
    def __init__(self, model_name="pucpr/biobertpt-all"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        print(f"Modelo preparado para uso de {self.device}")
    
    def train(self, 
              X: pd.Series,
              y: pd.Series,
              epochs: int = 8,
              batch_size: int = 8,
              learning_rate: float = 1e-5,
              test_size: float = 0.2,
              random_state: int = 42,
              max_length: int = 512,
              use_dropout: bool = True,
              hidden_dropout: float = 0.1,
              attention_dropout: float = 0.1,
              patience: int = 3,
              min_delta: float = 0.001):
        
        # Split dos dados
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        
        # Carregar tokenizer e modelo
        self.tokenizer = BertTokenizer.from_pretrained(self.model_name)
        
        if use_dropout:
            self.model = BertForSequenceClassification.from_pretrained(
                self.model_name,
                num_labels=2,
                hidden_dropout_prob=hidden_dropout,
                attention_probs_dropout_prob=attention_dropout
            )
        else:
            self.model = BertForSequenceClassification.from_pretrained(
                self.model_name,
                num_labels=2
            )
        
        # Tokenização
        train_encodings = self.tokenizer(
            list(X_train),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        test_encodings = self.tokenizer(
            list(X_test),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        
        # DataLoaders
        train_dataset = TensorDataset(
            train_encodings["input_ids"],
            train_encodings["attention_mask"],
            torch.tensor(y_train.values)
        )
        train_sampler = RandomSampler(train_dataset)
        train_dataloader = DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size)
        
        test_dataset = TensorDataset(
            test_encodings["input_ids"],
            test_encodings["attention_mask"],
            torch.tensor(y_test.values)
        )
        test_sampler = SequentialSampler(test_dataset)
        test_dataloader = DataLoader(test_dataset, sampler=test_sampler, batch_size=batch_size)
        
        # Otimizador
        optimizer = AdamW(self.model.parameters(), lr=learning_rate)
        self.model.to(self.device)
        
        # Listas para armazenar métricas
        train_losses = []
        test_losses = []
        best_test_loss = float('inf')
        epochs_without_improvement = 0
        best_model_state = None
        
        # Loop de treinamento
        for epoch in tqdm(range(epochs)):
            print(f'\n=== Epoch {epoch + 1}/{epochs} ===')
            
            # TREINAMENTO
            self.model.train()
            total_train_loss = 0
            
            for batch in train_dataloader:
                batch = tuple(t.to(self.device) for t in batch)
                input_ids, attention_mask, labels = batch
                
                optimizer.zero_grad()
                outputs = self.model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                total_train_loss += loss.item()
                
                loss.backward()
                optimizer.step()
            
            avg_train_loss = total_train_loss / len(train_dataloader)
            train_losses.append(avg_train_loss)
            print(f"Average training loss: {avg_train_loss:.4f}")
            
            # VALIDAÇÃO
            self.model.eval()
            total_test_loss = 0
            predictions = []
            true_labels = []
            probabilities = []
            
            for batch in test_dataloader:
                batch = tuple(t.to(self.device) for t in batch)
                input_ids, attention_mask, labels = batch
                
                with torch.no_grad():
                    outputs = self.model(input_ids, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss
                    total_test_loss += loss.item()
                    
                    logits = outputs.logits
                    probs = torch.softmax(logits, dim=-1)
                    predictions.extend(torch.argmax(probs, dim=-1).tolist())
                    true_labels.extend(labels.tolist())
                    probabilities.extend(probs[:, 1].tolist())
            
            avg_test_loss = total_test_loss / len(test_dataloader)
            test_losses.append(avg_test_loss)
            print(f"Average test loss: {avg_test_loss:.4f}")
            
            # Calcular métricas de validação
            accuracy = accuracy_score(true_labels, predictions)
            precision = precision_score(true_labels, predictions, zero_division=0)
            recall = recall_score(true_labels, predictions, zero_division=0)
            f1 = f1_score(true_labels, predictions, zero_division=0)
            
            print(f"Validation Metrics - Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
            
            # Registrar métricas no MLflow
            
            mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
            mlflow.log_metric("test_loss", avg_test_loss, step=epoch)
            mlflow.log_metric("val_accuracy", accuracy, step=epoch)
            mlflow.log_metric("val_precision", precision, step=epoch)
            mlflow.log_metric("val_recall", recall, step=epoch)
            mlflow.log_metric("val_f1_score", f1, step=epoch)
            
            # Early stopping
            if avg_test_loss < best_test_loss - min_delta:
                best_test_loss = avg_test_loss
                epochs_without_improvement = 0
                best_model_state = deepcopy(self.model.state_dict())
                print(f"✓ Novo melhor modelo! Test loss: {best_test_loss:.4f}")
            else:
                epochs_without_improvement += 1
                print(f"✗ Sem melhoria. Épocas sem melhoria: {epochs_without_improvement}/{patience}")
                
                if epochs_without_improvement >= patience:
                    print(f"\n⚠ Early stopping acionado após {epoch + 1} épocas")
                    break
        
        # Restaurar o melhor modelo
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
            print("\n✓ Melhor modelo restaurado")
        
        # Avaliação final
        self.model.eval()
        final_predictions = []
        final_true_labels = []
        final_probabilities = []
        
        for batch in test_dataloader:
            batch = tuple(t.to(self.device) for t in batch)
            input_ids, attention_mask, labels = batch
            
            with torch.no_grad():
                outputs = self.model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                probs = torch.softmax(logits, dim=-1)
                final_predictions.extend(torch.argmax(probs, dim=-1).tolist())
                final_true_labels.extend(labels.tolist())
                final_probabilities.extend(probs[:, 1].tolist())
        
        # Métricas finais
        report = classification_report(final_true_labels, final_predictions, output_dict=True)
        print("\n" + "="*50)
        print("RELATÓRIO DE CLASSIFICAÇÃO FINAL")
        print("="*50)
        print(classification_report(final_true_labels, final_predictions))
        
        # Registrar métricas finais no MLflow
        
        mlflow.log_metric("final_accuracy", report['accuracy'])
        mlflow.log_metric("final_precision_class_0", report['0']['precision'])
        mlflow.log_metric("final_recall_class_0", report['0']['recall'])
        mlflow.log_metric("final_f1_score_class_0", report['0']['f1-score'])
        mlflow.log_metric("final_precision_class_1", report['1']['precision'])
        mlflow.log_metric("final_recall_class_1", report['1']['recall'])
        mlflow.log_metric("final_f1_score_class_1", report['1']['f1-score'])
        self._save_confusion_matrix(final_true_labels, final_predictions)
        self._save_roc_curve(final_true_labels, final_probabilities)
        self._save_precision_recall_curve(final_true_labels, final_probabilities)
        self._save_loss_curve(train_losses, test_losses)
        return self.model, X_test, y_test, final_predictions, final_probabilities
    
    def _save_confusion_matrix(self, true_labels, predictions):
        """Salva matriz de confusão"""
        cm = confusion_matrix(true_labels, predictions)
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap='Blues')
        plt.title('Matriz de Confusão')
        plt.colorbar()
        plt.xlabel('Predição')
        plt.ylabel('Valor Real')
        plt.xticks([0, 1], ['Negativo (0)', 'Positivo (1)'])
        plt.yticks([0, 1], ['Negativo (0)', 'Positivo (1)'])
        
        thresh = cm.max() / 2
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, format(cm[i, j], 'd'),
                        ha="center", va="center",
                        color="white" if cm[i, j] > thresh else "black")
        
        plt.tight_layout()
        plt.savefig('../Artefatos/confusion_matrix.png')
        mlflow.log_artifact('../Artefatos/confusion_matrix.png')
        plt.close()
    
    def _save_roc_curve(self, true_labels, probabilities):
        """Salva curva ROC"""
        fpr, tpr, _ = roc_curve(true_labels, probabilities)
        roc_auc = auc(fpr, tpr)
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('Taxa de Falsos Positivos')
        plt.ylabel('Taxa de Verdadeiros Positivos')
        plt.title('Curva ROC')
        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
        plt.savefig('../Artefatos/roc_curve.png')
        mlflow.log_artifact('../Artefatos/roc_curve.png')
        mlflow.log_metric("roc_auc", roc_auc)
        plt.close()
    
    def _save_precision_recall_curve(self, true_labels, probabilities):
        """Salva curva Precision-Recall"""
        precision, recall, _ = precision_recall_curve(true_labels, probabilities)
        
        plt.figure(figsize=(8, 6))
        plt.plot(recall, precision, color='blue', lw=2)
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Curva Precision-Recall')
        plt.grid(True, alpha=0.3)
        plt.savefig('../Artefatos/precision_recall_curve.png')
        mlflow.log_artifact('../Artefatos/precision_recall_curve.png')
        plt.close()
    
    def _save_loss_curve(self, train_losses, test_losses):
        """Salva curva de perda"""
        epochs_range = range(1, len(train_losses) + 1)
        
        plt.figure(figsize=(10, 6))
        plt.plot(epochs_range, train_losses, 'b-', label='Perda de Treinamento', marker='o')
        plt.plot(epochs_range, test_losses, 'r-', label='Perda de Validação', marker='s')
        plt.xlabel('Época')
        plt.ylabel('Perda')
        plt.title('Curva de Perda durante o Treinamento')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig('../Artefatos/loss_curve.png')
        mlflow.log_artifact('../Artefatos/loss_curve.png')
        plt.close()
    
    def infer(self, X: pd.Series, batch_size: int = 8) -> pd.DataFrame:
        """Realiza inferência em novos dados"""
        if not self.tokenizer or not self.model:
            print("Tokenizer ou modelo não definidos!")
            return None
        
        val_encodings = self.tokenizer(
            list(X),
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        
        inference_dataset = TensorDataset(
            val_encodings["input_ids"],
            val_encodings["attention_mask"]
        )
        inference_dataloader = DataLoader(inference_dataset, batch_size=batch_size)
        
        probs_list = []
        preds_list = []
        
        self.model.eval()
        for batch in inference_dataloader:
            batch = tuple(t.to(self.device) for t in batch)
            input_ids, attention_mask = batch
            
            with torch.no_grad():
                outputs = self.model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                probs = torch.softmax(logits, dim=-1)
                probs_list += probs[:, 1].tolist()
                preds_list += torch.argmax(probs, dim=-1).tolist()
        
        df_output = pd.DataFrame({
            "X": X,
            "y_infered": preds_list,
            "prob": probs_list
        })
        
        return df_output

## 9. Wrapper MLflow para Deploy

In [0]:
class BertMLflowWrapper(mlflow.pyfunc.PythonModel):
    """Wrapper para deploy do modelo BERT no MLflow"""
    
    def __init__(self, model_name, model, tokenizer, preprocess=None, device=None):
        self.model_name = model_name.upper()
        self.model_threshold = 0.5
        self.device = torch.device("cuda") if (device is None and torch.cuda.is_available()) else torch.device("cpu")
        self.model = model.to(self.device)
        self.tokenizer = tokenizer

        if preprocess is None:
            self._preprocess = lambda x: x
        else:
            self._preprocess = preprocess

    def preprocess(self, text):
        """
        Realiza o pré-processamento do texto fornecido.
        """
 
        text = self._preprocess(text)
        return text
    
    def load_context(self, context):
        """Carrega o contexto do modelo"""
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

     
    
    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        """
        Realiza predições e extrai embeddings para monitoramento de data drift.
        
        Retorna:
            DataFrame com colunas:
            - model_name: nome do modelo
            - flag: classe predita (0 ou 1)
            - score: probabilidade da classe positiva
            - embedding: vetor de 768 dimensões do token [CLS] para drift monitoring  # MODIFICADO: docstring atualizada
        """
        if params is not None:
            if "threshold" in params:
                self.model_threshold = params["threshold"]
        model_input["text"] = model_input["text"].apply(self.preprocess)
        texts = model_input["text"].tolist()
        self.model.eval()
        
        # Tokenização
        inputs = self.tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(self.device)
        
        with torch.no_grad():
            # MODIFICADO: adicionado output_hidden_states=True para extrair embeddings das camadas ocultas
            outputs = self.model(**inputs, output_hidden_states=True)
        
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        
        # ADICIONADO: Extrair embeddings do token [CLS] (primeira posição) da última camada oculta
        # Shape: (batch_size, sequence_length, hidden_size=768)
        # Pegamos [:, 0, :] para obter apenas o token [CLS]
        cls_embeddings = outputs.hidden_states[-1][:, 0, :].cpu().numpy()
        
        flags = torch.argmax(probabilities, dim=-1).tolist()
        scores = probabilities[:, 1].tolist()
        
        # Aplicar threshold
        if self.model_threshold != 0.5:
            flags = [1 if score >= self.model_threshold else 0 for score in scores]
        
        results = pd.DataFrame({
            "model_name": self.model_name,
            "flag": flags,
            "score": scores,
            "embedding": [emb.tolist() for emb in cls_embeddings]  # ADICIONADO: campo embedding para drift monitoring
        })
        
        return results

### 9.1. Uso dos Embeddings para Data Drift Monitoring

O wrapper `BertMLflowWrapper` agora retorna embeddings do token [CLS] (768 dimensões) em cada predição.
Esses embeddings podem ser usados para monitoramento de data drift da seguinte forma:

**Como usar os embeddings:**
1. Durante a inferência, os embeddings são automaticamente incluídos na coluna `embedding`
2. Armazene os embeddings junto com as predições no banco de dados
3. Use ferramentas de drift detection (Evidently, NannyML, etc.) para comparar distribuições

**Métricas recomendadas para drift:**
- **PSI (Population Stability Index)**: Mede mudança na distribuição
- **KL-Divergence**: Divergência entre distribuições de referência e produção
- **Wasserstein Distance**: Distância entre distribuições multivariadas
- **Cosine Similarity**: Similaridade média entre embeddings de referência e produção

**Exemplo de uso:**
```python
# Inferência com embeddings
model_input = pd.DataFrame({"text": ["Texto do laudo médico"]})
result = loaded_model.predict(model_input)
# result contém: model_name, flag, score, embedding (lista de 768 floats)

# Armazenar embeddings para drift monitoring
# embeddings_array = np.array(result['embedding'].tolist())
```

## 10. Carregamento e Preparação dos Dados

In [0]:
print(f"Carregando dados de: {dataset_table_name}")
df = spark.read.table(dataset_table_name).toPandas()

print(f"Shape dos dados: {df.shape}")
print(f"\nPrimeiras linhas:")
display(df.head())

Carregando dados de: datascience_prd.gold.dataset_anatomo_v2
Shape dos dados: (16965, 3)

Primeiras linhas:


CD_LAUDO DS_LAUDO Y_REAL 5.75210011E8 Exame....: IMUNO HISTOQUIMICA
Microscopia Óptica

10748267
LINFONODO SENTINELA
E-Caderina
NCH-38-DAKO-IR059-FLEX
POSITIVO
10748267
LINFONODO SENTINELA
HER2
Policlonal-DAKO-A0485
ESCORE 1+
10748267
LINFONODO SENTINELA
KI67
MIB-1-DAKO-IR626-FLEX
POSITIVO 15%
10748267
LINFONODO SENTINELA
RE (receptor de estrógeno)
EP1-DAKO-IR084-FLEX
POSITIVO 80% +++
10748267
LINFONODO SENTINELA
RP (receptor de progesterona)
 PgR636-DAKO-IR068-FLEX
POSITIVO 10% +
0600297564003
LINFONODO SENTINELA
VIDE COMENTÁRIO
CARCINOMA MAMÁRIO POSITIVO PARA RECEPTORES DE ESTRÓGENO E PROGESTERONA E ESCORE 1+ PARA HER-2. 
 
 -A POSITIVIDADE PARA O ANTICORPO E-CADERINA INDICA DIFERENCIAÇÃO DUCTAL PARA A NEOPLASIA.
 
 NOTAS 
 1. AVALIAÇÃO IMUNO-HISTOQUÍMICA DE RECEPTORES DE ESTRÓGENO E DE PROGESTERONA (ASCO/CAP, 2020): 
 - Resultado positivo: expressão nuclear em pelo menos 1% das células neoplásicas, independentemente da intensidade da imunocoloração; é importante salientar que há dados limitados quanto ao benefício da terapia endócrina para pacientes com tumores exibindo positividade de 1-10% para receptores de estrógeno/RE (baixa positividade ou "low positive"): há possível benefício, sendo os pacientes considerados elegíveis para tratamento endócrino. Contudo, dados de literatura sugerem que esses tumores são heterogêneos tanto em comportamento quanto em sua biologia e frequentemente tem perfis de expressão gênica mais similares aos tumores RE-negativos.
 - Resultado negativo: ausência de expressão nuclear ou expressão nuclear observada em <1% das células neoplásicas, independente da intensidade da imunocoloração.
 
 2. AVALIAÇÃO IMUNO-HISTOQUÍMICA DE HER2 (ASCO/CAP, 2018): 
 - Escore 0: ausência de imunocoloração ou imunocoloração incompleta/quase imperceptível (padrão membrana) em até 10% das células neoplásicas 
 - Escore 1+: imunocoloração incompleta/quase imperceptível (padrão membrana) em >10% das células neoplásicas 
 - Escore 2+: imunocoloração completa fraca a moderada (padrão membrana) em >10% das células ou intensa e completa em até 10% das células neoplásicas; necessidade de investigação de amplificação gênica por SISH ou FISH
 - Escore 3+: imunocoloração intensa e completa (padrão membrana) em >10% das células neoplásicas
 - Segundo estudo recente (DESTINY-Breast04), o conjugado anticorpo-droga fam-trastuzumab deruxtecan-nxki (T-DXd) dobrou a sobrevida livre de progressão em comparação com quimioterapia isolada em pacientes com câncer de mama metastático ¿HER2-low¿ ¿ ou seja, pacientes com baixa expressão de HER2 (escore 1+ e escore 2+ sem amplificação gênica), o que pode corresponder a quase 50% das pacientes com carcinoma de mama. O agente também estendeu a sobrevida global para pacientes com baixos níveis do receptor HER2, independentemente do status do receptor hormonal.
 
 3.CONTROLES POSITIVO E NEGATIVOS , INTERNOS EXTERNOS ATESTAM A FIDELIDADE DAS REAÇÕES.
 
 4-REFERÊNCIAS BIBLIOGRÁFICAS: 
 1. Wolff AC, Somerfield MR, Dowsett M, Hammond MEH, Hayes DF, McShane LM, Saphner TJ, Spears PA, Allison KH. Human Epidermal Growth Factor Receptor 2 Testing in Breast Cancer. Arch Pathol Lab Med. 2023 Sep 1;147(9):993-1000. doi: 10.5858/arpa.2023-0950-SA. PMID: 37303228.
 2. Wolff AC, Hammond MEH, Allison KH et al. Human Epidermal Growth Factor Receptor 2 Testing in Breast Cancer: American Society of Clinical Oncology/College of American Pathologists Clinical Practice Guideline Focused Update. Arch Pathol Lab Med. 2018;142(11):1364-1382. 
 3. WHO Classification of Tumours Editorial Board (Eds.): WHO Classification of Tumours. Breast Tumours. 5th Edition. IARC: Lyon, 2019.
 4. https://documents.cap.org/documents/Breast.Bmk_1.5.0.1.REL_CAPCP.pdf (CAP Cancer Protocols, Março de 2023)
 5. Modi S, Jacot W, Yamashita T, et al. Trastuzumab deruxtecan (T-DXd) vs treatment of physician¿s choice in patients with HER2-low unresectable and/or metastatic breast cancer: Results of DESTINY-Breast04, a random

In [0]:
# df['Y_REAL'] = df['DS_SEVERIDADE'].replace({'SEM TUMOR': 0, 'BENIGNO': 0,
#                                             'MALIGNO': 1,
#                                             'SEVERIDADE NÃO ENCONTRADA': -1}).astype('int64')

In [0]:
# Pré-processamento
sample_final = df.copy()
sample_final = sample_final[sample_final['Y_REAL'].isin([1, 0])].reset_index(drop=True)
#sample_final = df_downsample(sample_final, 'Y_REAL', True, random_seed).reset_index(drop=True) Aqui não usaremos porque no OCI está sem e já em produção

#sample_final_lst = preprocess_laudo_lst(func_column_to_lst(sample_final, 'DS_LAUDO'))
sample_final['DS_LAUDO'] = sample_final['DS_LAUDO'].apply(lambda x: preprocess_bert(x))

# Separação de features e target
X = sample_final['DS_LAUDO']
y = sample_final['Y_REAL'].astype('int64')

print(f"\nDistribuição das classes:")
print(y.value_counts())
print(f"\nTotal de amostras: {len(X)}")


Distribuição das classes:
Y_REAL
1    12431
0     4534
Name: count, dtype: int64

Total de amostras: 16965


## 11. Loop de Treinamento com Grid Search

In [0]:
# Limpar strings para nome da run
c_foundation_model_name = clean_string(str(foundation_model_name))
c_learning_rate = clean_string(str(learning_rate))

# Nome da run
run_name = f"{formatted_datetime}_BS{batch_size}_LR{c_learning_rate}_TL{token_max_length}_EP{epochs}"

print("\n" + "="*100)
print(f"INICIANDO TREINAMENTO: {run_name}")
print("="*100)

# Logar dataset no MLflow
mlflow_dataset = mlflow.data.from_pandas(
    df, source=dataset_table_name, name=dataset_table_name, targets="Y_REAL"
)

with mlflow.start_run(run_name=run_name):
    mlflow.log_input(mlflow_dataset, context="training")
    run_id = mlflow.active_run().info.run_id
    run_details = mlflow.get_run(run_id)
    print(f"Run ID: {run_id}")
    print(f"Run Name: {run_details.info.run_name}")
    print(f"Artifact URI: {run_details.info.artifact_uri}")
    
    # Registrar parâmetros do ambiente
    mlflow.log_param("python_version", python_version)
    mlflow.log_param("numpy_version", numpy_version)
    mlflow.log_param("pandas_version", pandas_version)
    mlflow.log_param("scikit_learn_version", scikit_learn_version)
    mlflow.log_param("mlflow_version", mlflow_version)
    mlflow.log_param("pytorch_version", pytorch_version)
    mlflow.log_param("pytorch_version_cuda", pytorch_version_cuda)
    mlflow.log_param("pytorch_version_cudnn", pytorch_version_cudnn)
    mlflow.log_param("transformers_version", transformers_version)
    mlflow.log_param("cuda_available", cuda_available)
    mlflow.log_param("cuda_version", cuda_version)
    mlflow.log_param("device", str(device))
    
    # Registrar parâmetros do modelo

    mlflow.log_param("model_name", model_name)
    mlflow.log_param("random_seed", random_seed)
    mlflow.log_param("test_size", test_size)
    mlflow.log_param("use_validation", use_validation)
    mlflow.log_param("validation_size", validation_size)
    mlflow.log_param("foundation_model_name", foundation_model_name)
    mlflow.log_param("batch_size", batch_size)
    mlflow.log_param("learning_rate", learning_rate)
    mlflow.log_param("epochs", epochs)
    mlflow.log_param("token_max_length", token_max_length)
    mlflow.log_param("use_dropout", use_dropout)
    mlflow.log_param("hidden_dropout", hidden_dropout)
    mlflow.log_param("attention_dropout", attention_dropout)
    mlflow.log_param("use_scheduler", use_scheduler)
    mlflow.log_param("use_weights", use_weights)
    mlflow.log_param("patience", patience)
    mlflow.log_param("min_delta", min_delta)
    
    mlflow.log_param("total_samples", len(X))
    mlflow.log_param("class_0_count", int(y.value_counts()[0]))
    mlflow.log_param("class_1_count", int(y.value_counts()[1]))

# Instanciar e treinar o modelo
    bert_model = BERTBinaryClassification(
        model_name=foundation_model_name
    )

    trained_model, X_test, y_test, predictions, probabilities = bert_model.train(
        X=X,
        y=y,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        test_size=test_size,
        random_state=random_seed,
        max_length=token_max_length,
        use_dropout=use_dropout,
        hidden_dropout=hidden_dropout,
        attention_dropout=attention_dropout,
        patience=patience,
        min_delta=min_delta
    )

# Registrar o modelo no MLflow com wrapper
    print("\n" + "="*50)
    print("REGISTRANDO MODELO NO MLFLOW")
    print("="*50)
    
    # Criar wrapper
    model_class = BertMLflowWrapper(
        model_name=model_name,
        model=trained_model,
        tokenizer=bert_model.tokenizer,
        preprocess=preprocess_bert,
        device=device
    )
    
    # Criar signature
    input_texts = ["Texto de exemplo para inferência"]
    model_input = pd.DataFrame({"text": input_texts})
    model_params = {"threshold": 0.5}
    
    model_output = pd.DataFrame({
        "model_name": [model_name.upper()],
        "flag": [0],
        "score": [0.5],
        "embedding": [[0.0] * 768]  # ADICIONADO: Vetor de 768 dimensões para embeddings [CLS]
    })
    
    model_output = model_output.astype({
        "flag": "int64",
        "score": "float64",
    })
    
    signature = infer_signature(
        model_input=model_input,
        model_output=model_output,
        params=model_params
    )
    
    print(f"\nSignature do modelo:")
    print(signature)
    
    # Dependências
    pip_requirements = [
        f"torch=={pytorch_version}",
        f"transformers=={transformers_version}",
        "unidecode",
        f"cloudpickle=={cloudpickle_version}"
    ]
    
    # Registrar modelo
    mlflow.pyfunc.log_model(
        artifact_path=model_name,
        python_model=model_class,
        input_example=model_input,
        signature=signature,
        pip_requirements=pip_requirements
    )
    
    print("✓ Modelo registrado com sucesso!")
    
    # Finalizar run
    mlflow.end_run()
    print("\n✓ Run MLflow finalizada")

    # Limpar memória
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.ipc_collect()

    print("\n" + "="*100)
    print("TREINAMENTO CONCLUÍDO")
    print("="*100 + "\n")


INICIANDO TREINAMENTO: 06_02_26_122513_BS32_LR1e_05_TL512_EP2


/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'datascience_prd.gold.dataset_anatomo_v2'. Exception: 
  return _dataset_source_registry.resolve(
/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/databricks/python/lib/python3.12/site-packages/databricks/sdk/service/jobs.py:60: SyntaxWarning: invalid escape sequence '\.'
  """The sequence number of this run attempt for a triggered job run. The initial attempt of a run
/databricks/python/lib/python3.12/site-packages/databricks/sdk/service/jobs.py:

Run ID: f5945b58beaf48a9b78cc9207cf5252a
Run Name: 06_02_26_122513_BS32_LR1e_05_TL512_EP2
Artifact URI: dbfs:/databricks/mlflow-tracking/4472395647116746/f5945b58beaf48a9b78cc9207cf5252a/artifacts
Modelo preparado para uso de cuda


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...


RELATÓRIO DE CLASSIFICAÇÃO FINAL
              precision    recall  f1-score   support

           0       0.86      0.87      0.87       896
           1       0.95      0.95      0.95      2497

    accuracy                           0.93      3393
   macro avg       0.91      0.91      0.91      3393
weighted avg       0.93      0.93      0.93      3393


REGISTRANDO MODELO NO MLFLOW


/local_disk0/.ephemeral_nfs/cluster_libraries/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/02/06 12:37:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Signature do modelo:
inputs: 
  ['text': string (required)]
outputs: 
  ['model_name': string (required), 'flag': long (required), 'score': double (required), 'embedding': Array(double) (required)]
params: 
  ['threshold': double (default: 0.5)]



🔗 View Logged Model at: https://adb-420884346733669.9.azuredatabricks.net/ml/experiments/4472395647116746/models/m-444ff4d0bcfa4873b5be9dd836ea6446?o=420884346733669
2026/02/06 12:37:41 INFO mlflow.pyfunc: Validating input example against model signature


✓ Modelo registrado com sucesso!

✓ Run MLflow finalizada

TREINAMENTO CONCLUÍDO



In [0]:
# Registrar no Model Registry
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_uri = f"runs:/{run_id}/{model_name}"

model_details = mlflow.register_model(
    model_uri=model_uri,
    name=f"datascience_prd.models.{model_name.lower()}_classifier"
)

print(f"✅ Modelo versão {model_details.version} registrado")

Registered model 'datascience_prd.models.anatomopathologicals_classifier' already exists. Creating a new version of this model...
2026/02/06 12:37:47 WARNING mlflow.tracking._model_registry.fluent: Run with id f5945b58beaf48a9b78cc9207cf5252a has no artifacts at artifact path 'ANATOMOPATHOLOGICALS', registering model based on models:/m-444ff4d0bcfa4873b5be9dd836ea6446 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '12' of model 'datascience_prd.models.anatomopathologicals_classifier': https://adb-420884346733669.9.azuredatabricks.net/explore/data/models/datascience_prd/models/anatomopathologicals_classifier/version/12?o=420884346733669


✅ Modelo versão 12 registrado


In [0]:
# Promover para production
client.set_registered_model_alias(
    name=f"datascience_prd.models.{model_name.lower()}_classifier",
    alias="production",
    version=model_details.version
)

print(f"🚀 Modelo v{model_details.version} promovido para PRODUCTION")

🚀 Modelo v12 promovido para PRODUCTION


In [0]:
# Test model locally with correct MLflow model URI
print("🧪 Testando modelo localmente...")

model_uri = (
    f"models:/datascience_prd.models.{model_name.lower()}_classifier@production"
)
loaded_model = mlflow.pyfunc.load_model(model_uri)

test_data = pd.DataFrame({"text": input_texts})

result = loaded_model.predict(test_data)
print("\n📊 Resultado do teste:")
print(result)

🧪 Testando modelo localmente...



📊 Resultado do teste:
             model_name  ...                                          embedding
0  ANATOMOPATHOLOGICALS  ...  [0.10900523513555527, -0.40243586897850037, 0....

[1 rows x 4 columns]
